# 👗 Predict Next Fashion Trends

Designing new products for a fashion line takes time. Taking finished products from design to market can have lead times of a year or more! Therefore, the ability to predict upcoming trends in the market has the potential to avoid financial losses and wastage.

With that in mind, let's try using social network signals to anticipate and predict the next big fashion trends!

## 🎯 Learning Objectives

- Time series forecasting fundamentals
- Identifying trends and seasonality
- Seasonal decomposition
- Naive forecasting baseline
- Holt-Winters exponential smoothing
- Time series metrics (MAE, MASE, SMAPE)

## 🏢 Business Context

Fashion companies need to predict trends 12+ months in advance.
Using social media signals, we can forecast which styles will be popular.

## 📊 The Data

- **Source**: Social media popularity index (2015-2019)
- **Frequency**: Weekly data (52 weeks/year)
- **Features**: Different clothing items
- **Task**: Forecast next year's trends


Let's dive in! 👇

## 📦 Package Installation

The packages below are not yet included in the standard lewagon setup. Run this cell once before starting the challenge.


In [ ]:
%pip install "scikit-base==0.13.1" "sktime==0.40.1"


## Load data

a) Download the [`tendances.csv`](https://drive.google.com/uc?export=download&id=15xv2daQ03RnTL8fHksHxQFC3yrU7VFVg) file, create a `data` directory in this challenge and save it there.

<details>
<summary markdown="span">Not sure where your downloaded file went?</summary>

You can move it manually into the `data` folder, or run one of these in your terminal:

- **macOS/Linux:** `mv ~/Downloads/tendances.csv data/`
- **Windows (WSL):** `mv /mnt/c/Users/YOUR_USERNAME/Downloads/tendances.csv data/`

</details>

Load the data into a `df` DataFrame. 

Use the parameter `index_col=0` to set the first column as the index.

What is the time frequency of this data? (Daily, weekly, monthly, yearly?)

In [ ]:
import pandas as pd

# Data is on a weekly basis
df = pd.read_csv('data/tendances.csv', index_col=0)
df

<details>
<summary>Answer</summary>

The data is **weekly** - there are 52 observations per year.

</details>

Let's take a look at the data. 

The columns represent different types of clothing and the rows represent dates between 2015 and 2019. 

The values correspond to an internal index related to their popularity on social networks.

We're trying to predict future values for each type of clothing.

b) Plot all the series on the same chart with `plotly.express`

In [ ]:
import plotly.express as px

px.line(df)

c) Do you notice any clear trends (upwards or downwards) for any of the clothing items?

Indeed, we can see a few major trends appearing:

- lace-up shoes (`shoes_laceup`), denim (`pants_denim`) are on the rise globally

- tanks with close round collars (`top_tanksleeve_tshirtnect`) and square glasses (`eyewear_squaredglassesshape`) are decreasing globally

- dresses or shirts with thin straps (`body_spaghettistraps`) are increasing year after year, but only in summer; they remain stable in the other seasons

d) Can you identify any **seasonal patterns** in the data?

**What is seasonality?** Patterns that repeat at regular intervals (e.g., yearly, quarterly).

For example:
- Do certain items spike every summer?
- Are there winter vs. summer favorites?
- Do patterns repeat year after year?

<details>
<summary>Answer</summary>

Yes! All series show patterns that recur year after year. 

The data is weekly with clear **52-week cycles** (one year). For example:
- Spaghetti straps peak every summer
- Denim has two seasonal peaks (spring and fall)

</details>

e) There's a quick way to confirm our intuition about the seasonality of these fashion trends. We can decompose time series data to see the `trend` + `seasonality` + `residuals`.

Execute the cell below to see the results for denim trousers.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

# decompose time series
r = seasonal_decompose(df['pants_denim'], period=52)

# save plot to fig variable
fig = r.plot()
# set custom dimensions
fig.set_figwidth(12)
fig.set_figheight(8)
plt.tight_layout()

# Set x-axis ticks every 26 weeks (approximately every 6 months)
for ax in fig.get_axes():
    ax.set_xticks(list(range(0,209,26)))

f) Interpret the seasonal decomposition plot for denim trousers.

**How to read the plot:**
- **Observed**: The original data
- **Trend**: Long-term direction (up, down, or stable)
- **Seasonal**: Repeating yearly pattern
- **Residual**: Random noise (what's left over)

**Questions:**

1. **Seasonality**: When during the year are denim trousers most popular? 
   (Look at the "Seasonal" line - what months have peaks?)

2. **Trend**: What happened to denim's popularity between 2015 and 2018? 
   (Look at the "Trend" line - rising, falling, or stable?)

<details>
<summary>Answer</summary>

**Seasonality:** Denim has two seasonal peaks - **spring and fall** 
(the two recurring "bumps" in the seasonal component).

**Trend:** Denim popularity **increased** between 2015 and 2018, 
then **stabilized** around 2018-2019.

</details>

**Challenge:** Try modifying the code above to decompose a different clothing item. 
Can you predict its seasonal pattern before running the code?

## Preprocessing Data

a) Prepare the data for time series modeling. Time series models in `sktime` require specific data formats:

1. **Period Index**: Index must be `PeriodIndex` with weekly frequency
   - This tells the model: "each row = 1 week"
   - Period type handles time math correctly (week + 1 = next week)

1. **Datetime Column**: Keep a `date` column for plotting
   - Period index is great for modeling
   - But datetime works better for `plotly` charts

First, check the current index type: `df.index.dtype`

In [ ]:
# YOUR CODE HERE

Looks like we'll need to turn our index into a date type and change the frequency to weekly

<details>
<summary>Hint</summary>

You can't use `pd.to_datetime` on the index directly, you'll need to:

- create a new column based on the index values
- make that column a datetime
- replace the index with the new values!

For swapping the frequency, let's look at the documentation here: [to_period()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_period.html)

</details>

**Note:** If you need to re-run this cell, first re-run the data loading cell above to reset the DataFrame.

<details>
<summary>Why?</summary>

Once the index is converted to Period type, it can't be converted again. Re-loading the data resets it to the original format.

</details>

In [ ]:
# Create a new column 'date' containing the df.index cast into datetime format
pass  # YOUR CODE HERE

# Set index to period type (weekly frequency)
pass  # YOUR CODE HERE

df

b) Now we'll need to split the train and test data. We will use the last year of data as our test set (`df_test`) and the rest for training (`df_train`).

We're going to use the `temporal_train_test_split` method. What should you put in the `test_size` parameter?

In [ ]:
from sktime.forecasting.model_selection import temporal_train_test_split

# 52 weeks = 1 year
pass  # YOUR CODE HERE

In [ ]:
# Make independent copies to avoid SettingWithCopyWarning
df_train = df_train.copy()
df_test = df_test.copy()

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'data_loading',
    n_rows=len(df),
    index_is_period='period' in str(df.index.dtype).lower(),
    test_size=len(df_test)
)
result.write()
print(result.check())

## Modeling

### Baseline Model: Naive Forecaster

For the rest of this exercise, we'll focus on one clothing item: 
`top_tanksleeve_tshirtneck`.

**What is a Naive Seasonal Model?**

The simplest time series forecast: 
> "Next year will be exactly like last year"

For weekly data with yearly seasonality:
- Prediction for week 1 of 2019 = week 1 of 2018
- Prediction for week 2 of 2019 = week 2 of 2018
- ...and so on

This is surprisingly effective! It's hard to beat for seasonal data.

a) Train a `NaiveForecaster` on the training data.

Use `sp=52` (seasonal period = 52 weeks = 1 year).

Predict the next 52 weeks and store the results in a variable called `naive_preds`.

In [ ]:
# YOUR CODE HERE

b) In `df_test` variable, create a new column `naive_model` with naive prediction values.

Visualize predicted and real data with plotly express (y parameter of px.line() function can be a list of columns names)

In [ ]:
# YOUR CODE HERE

c) Calculate **MASE** (Mean Absolute Scaled Error) for the naive model.

**What is MASE?**

MASE compares your model to a naive seasonal baseline:
- **MASE = 1.0**: Your model is as good as naive
- **MASE < 1.0**: Your model beats naive 
- **MASE > 1.0**: Your model is worse than naive 

Since we're testing a naive model, we expect MASE ≈ 1.0.

Save the result as `naive_MASE`.

In [ ]:
# YOUR CODE HERE

<details>
    <summary><i>Response</i></summary>
MASE or Mean Absolute Scaled Error is the mean absolute error of the forecasted values divided by the mean absolute error if we had forecasted using a naive model (read: if we had just taken the same values as the previous season). Here we forecasted using a naive model, so we expected a value close to 1 anyway.
</details>



d) Calculate **MAE** and **SMAPE** metrics for the naive model.

**Metrics Explained:**

- **MAE** (Mean Absolute Error): Average absolute difference between predictions 
  and actual values. Lower is better. Scale: same units as data.

- **SMAPE** (Symmetric Mean Absolute Percentage Error): Percentage error. 
  Lower is better. Scale: 0-100%.

Save results as `naive_MAE` and `naive_SMAPE`.

In [ ]:
# YOUR CODE HERE

<details>
<summary><i>Why is SMAPE so high?</i></summary>

The MAE is about 0.0005. Since the time series ranges from 0 to 0.005, 
this error is about **10% of the typical amplitude** - not bad!

SMAPE is around 50% - much higher than we'd expect given the MAE. Why?

SMAPE calculates relative error:

```
SMAPE = |actual - predicted| / average(actual, predicted)
```

When values are very **small** (near zero), even tiny absolute errors become 
huge percentage errors.

**Example:**
- Actual: 0.0005
- Predicted: 0.0010
- Absolute error: 0.0005 (small!)
- Average: 0.00075
- SMAPE: 0.0005 / 0.00075 = **67%** (looks huge!)

**Takeaway:** SMAPE can be misleading for time series with values near zero. 
Always look at MAE too!

</details>

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'naive_model',
    naive_exists='naive_model' in locals(),
    naive_pred_length=len(naive_preds) if 'naive_preds' in locals() else 0,
    naive_mae=naive_MAE if 'naive_MAE' in locals() else None
)
result.write()
print(result.check())

### Exponential Smoothing Model: Holt-Winters

So, it's time to move on from our naive model to a more advanced model. Let's try the Holt-Winters model.
In this model, we start from a base level (the last value of our train data, before we start forecasting), and complete it with a trend component and a seasonal component.

e) **Choose model parameters** for Holt-Winters.

Holt-Winters models require two parameters: **trend** and **seasonality**.

Each can be **additive** or **multiplicative**:

**Additive:** Changes by constant amounts
- Trend: +100 units per year
- Seasonality: Same peak-to-trough difference each year

**Multiplicative:** Changes by constant percentages  
- Trend: +10% per year
- Seasonality: Peak-to-trough difference scales with overall level

Look at the plot for `top_tanksleeve_tshirtneck`:

Trend approaching zero? Use multiplicative (prevents negative values). Constant increases/decreases? Use additive

Seasonality with fixed amplitude year-over-year? Use additive. Amplitude grows/shrinks with trend? Use multiplicative

Choose `trend='add'` or `'mul'` and `seasonal='add'` or `'mul'`. Justify your choice.

<details>
<summary>Suggested Answer</summary>

**Recommended:**
- `trend='mul'` - Series decreasing toward zero; multiplicative prevents negative predictions
- `seasonal='add'` - Seasonal amplitude appears constant year-over-year
</details>

f) Apply a Holt-Winters model on this time series with trend and seasonal options.

Save the predictions in a variable called `holt_winters_preds`

In [ ]:
from sktime.forecasting.exp_smoothing import ExponentialSmoothing

pass  # YOUR CODE HERE

Display the predictions.

In [ ]:
# create a new column "holt_winters" with holt_winters predictions
pass  # YOUR CODE HERE

# visualize data
pass  # YOUR CODE HERE

g) Calculate error metrics for this new model (MAE, MASE and SMAPE) as you did questions c) and d).

Do you get a better performance? Interpret results.

Save the results to the following variables: 

- `holt_winters_MAE` 
- `holt_winters_SMAPE`
- `holt_winters_MASE`

In [ ]:
# YOUR CODE HERE

Predictions are very close to reality. We also observe an improvement according to all our metrics: this model is indeed an improvement compared to the naive model.

The MASE allows us to quantify specifically the improvement compared to the naive model: with a value of `0.68`, we can verify that our Holt-Winters model is 1/3 better than a naive model.

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'holt_winters',
    hw_exists='holt_winters' in locals(),
    hw_pred_length=len(holt_winters_preds) if 'holt_winters_preds' in locals() else 0,
    hw_mase=holt_winters_MASE if 'holt_winters_MASE' in locals() else None
)
result.write()
print(result.check())

## 🏆 Model Comparison

Now let's compare our two models side-by-side.

In [ ]:
# Create comparison table
comparison = pd.DataFrame({
    'Model': ['Naive', 'Holt-Winters'],
    'MAE': [naive_MAE, holt_winters_MAE],
    'MASE': [naive_MASE, holt_winters_MASE],
    'SMAPE (%)': [naive_SMAPE * 100, holt_winters_SMAPE * 100]
})

comparison

In [ ]:
# Visualize both predictions
fig = px.line(
    df_test,
    x="date",
    y=[cloth, "naive_model", "holt_winters"],
    title="Model Comparison: Naive vs Holt-Winters"
)
fig.show()

## 🏁 Conclusion

### 🏆 Key Takeaways

1. **Seasonality Matters**: Fashion trends show clear 52-week cycles
2. **Holt-Winters Wins**: 33% improvement over naive baseline (MASE = 0.68)
3. **Model Choice**: Multiplicative trend + additive seasonality worked best

### Business Insights

For `top_tanksleeve_tshirtneck`:
- Clear seasonal pattern (popular in summer, low in winter)
- Multiplicative trend captures decreasing popularity
- Predictions suggest continued decline in 2019

**Recommendation**: Focus design resources on rising trends (denim, lace-up shoes)

### What We Learned

- **Time series decomposition** reveals hidden patterns
- **Naive models** provide strong baselines (hard to beat!)
- **Holt-Winters** captures trend + seasonality elegantly
- **Multiple metrics** give complete picture (MAE, MASE, SMAPE)

### 📚 Resources

- [Sktime Documentation](https://www.sktime.net/en/stable/)
- [Holt-Winters Explained](https://otexts.com/fpp2/holt-winters.html)
- [Time Series Metrics](https://www.sktime.net/en/stable/api_reference/performance_metrics.html)